In [ ]:
from libraries import *
from parameters import *
from util import *

import pandas as pd
import patsy
import statsmodels.api as sm
from scipy import sparse

adata = sc.read_h5ad("./../../Data/ComboScreen.h5ad")

adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return 'and'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)
adata=adata[~((adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) == 2) & (adata.obs['NTC'] == 1)),]
adata.obs = adata.obs.drop(columns=['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776'], errors="ignore")

onehot = pd.get_dummies(adata.obs["perturbation"])
onehot = onehot.drop(columns=[c for c in ["None", "NTC"] if c in onehot.columns])
adata.obs = pd.concat([adata.obs, onehot], axis=1)


pert_cols = [
    c for c in onehot.columns
    if c not in ["NTC"] and c not in ["time_point"]
]

adata.obs["time_point"] = pd.Categorical(
    adata.obs["time_point"],
    categories=["day04", "day10"]
)

P = " + ".join(pert_cols)

formula = f"""
    C(time_point)
  + {P}
  + C(time_point):({P})
"""


sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)

X = patsy.dmatrix(formula, adata.obs, return_type="dataframe")


In [ ]:
X = patsy.dmatrix(formula, adata.obs, return_type="dataframe")
terms = X.columns.tolist()

# interaction terms to optionally save separately
interaction_prefix = "C(time_point)[T.day10]:"
interaction_terms = [t for t in terms if t.startswith(interaction_prefix)]
interaction_terms += [t for t in terms if t.endswith(":C(time_point)[T.day10]")]
interaction_terms = sorted(set(interaction_terms))

Y = adata.X
if sparse.issparse(Y):
    Y = Y.tocsr()

genes = adata.var_names.to_list()

outdir = "./ols_all_genes_results"
os.makedirs(outdir, exist_ok=True)

all_terms_path = os.path.join(outdir, "all_genes__all_terms.csv")
interactions_path = os.path.join(outdir, "all_genes__pert_time_interactions_only.csv")

# Write headers once
pd.DataFrame(columns=["gene", "term", "beta", "se", "t", "pval"]).to_csv(all_terms_path, index=False)
pd.DataFrame(columns=["gene", "term", "perturbation", "beta", "se", "t", "pval"]).to_csv(interactions_path, index=False)

def get_gene_y(g_idx: int) -> np.ndarray:
    if sparse.issparse(Y):
        return np.asarray(Y[:, g_idx].toarray()).ravel()
    return np.asarray(Y[:, g_idx]).ravel()

# ---- Batch buffering settings
BATCH_N_GENES = 500
buffer_all = []
buffer_int = []
batch_start = 0

def write_batch(batch_start, batch_end, buffer_all, buffer_int):
    all_path = os.path.join(
        outdir, f"all_genes__all_terms__{batch_start:05d}_{batch_end:05d}.csv"
    )
    int_path = os.path.join(
        outdir, f"all_genes__pert_time_interactions__{batch_start:05d}_{batch_end:05d}.csv"
    )

    pd.concat(buffer_all, ignore_index=True).to_csv(all_path, index=False)

    if buffer_int:
        pd.concat(buffer_int, ignore_index=True).to_csv(int_path, index=False)

    print(f"Saved genes {batch_start}–{batch_end}")
    
    
for g_idx, g in enumerate(genes):
    print(g_idx)
    print(g)
    y = get_gene_y(g_idx)
    fit = sm.OLS(y, X).fit()

    # ---- all terms
    df_gene = pd.DataFrame({
        "gene": g,
        "term": fit.params.index,
        "beta": fit.params.values,
        "se": fit.bse.values,
        "t": fit.tvalues.values,
        "pval": fit.pvalues.values,
    })
    buffer_all.append(df_gene)

    # ---- interaction-only
    if interaction_terms:
        df_int = df_gene[df_gene["term"].isin(interaction_terms)].copy()
        if not df_int.empty:
            df_int["perturbation"] = df_int["term"].str.replace(
                interaction_prefix, "", regex=False
            )
            df_int = df_int[
                ["gene", "term", "perturbation", "beta", "se", "t", "pval"]
            ]
            buffer_int.append(df_int)

    # ---- write every 500 genes
    if (g_idx + 1) % BATCH_N_GENES == 0:
        batch_end = g_idx
        write_batch(batch_start, batch_end, buffer_all, buffer_int)

        buffer_all = []
        buffer_int = []
        batch_start = g_idx + 1

# ---- write remaining genes
if buffer_all:
    write_batch(batch_start, len(genes) - 1, buffer_all, buffer_int)
